In [171]:
import pandas as pd

df = pd.read_excel('C:/Users/egor2/trainee_de/task2_data/task_2_data_ex.xlsx')
print("rows:", len(df))
print(df)

df_agg = df.groupby([
    'plant_id', 'year', 'produced_material', 'component_material',
    'produced_material_release_type', 'produced_material_production_type',
    'component_material_release_type', 'component_material_production_type'
]).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()

print("Rows after aggregation:", len(df_agg))

all_materials = set(df_agg['produced_material'])
all_components = set(df_agg['component_material'])
fin_materials = all_materials - all_components
print("FIN materials:", fin_materials)
print("Number of FIN materials:", len(fin_materials))

fin_material_combinations = df_agg[df_agg['produced_material'].isin(fin_materials)][['plant_id', 'year', 'produced_material']].drop_duplicates()
print(fin_material_combinations)

release_type_dict = df_agg[['produced_material', 'produced_material_release_type']].drop_duplicates('produced_material').set_index('produced_material')['produced_material_release_type'].to_dict()
prod_type_dict = df_agg[['produced_material', 'produced_material_production_type']].drop_duplicates('produced_material').set_index('produced_material')['produced_material_production_type'].to_dict()
comp_release_type_dict = df_agg[['component_material', 'component_material_release_type']].drop_duplicates('component_material').set_index('component_material')['component_material_release_type'].to_dict()
comp_prod_type_dict = df_agg[['component_material', 'component_material_production_type']].drop_duplicates('component_material').set_index('component_material')['component_material_production_type'].to_dict()
components_dict = df_agg.groupby('produced_material')['component_material'].apply(list).to_dict()

prod_quantity_dict = df_agg.groupby(['produced_material', 'year', 'plant_id'])['produced_material_quantity'].sum().to_dict()
comp_quantity_dict = df_agg.set_index(['produced_material', 'component_material', 'year', 'plant_id'])['component_material_quantity'].to_dict()

def build_hierarchy(fin_material, current_material, result_list, plant, year):
    components = components_dict.get(current_material, [])
    for component in components:
        try:
            row = {
                'plant': plant,
                'year': year,
                'fin_material_id': fin_material,
                'fin_material_release_type': release_type_dict.get(fin_material, 'Unknown'),
                'fin_material_production_type': prod_type_dict.get(fin_material, None),
                'fin_production_quantity': prod_quantity_dict.get((fin_material, year, plant), 0),
                'prod_material_id': current_material,
                'prod_material_release_type': release_type_dict.get(current_material, 'Unknown'),
                'prod_material_production_type': prod_type_dict.get(current_material, None),
                'prod_material_production_quantity': prod_quantity_dict.get((current_material, year, plant), 0),
                'component_id': component,
                'component_material_release_type': comp_release_type_dict.get(component, 'Unknown'),
                'component_material_production_type': comp_prod_type_dict.get(component, None),
                'component_consumption_quantity': comp_quantity_dict.get((current_material, component, year, plant), 0)
            }
            result_list.append(row)
            build_hierarchy(fin_material, component, result_list, plant, year)
        except Exception as e:
            print(f"Error in build_hierarchy for fin_material {fin_material}, component {component}: {e}")


result_list = []
for _, row in fin_material_combinations.iterrows():
    plant = row['plant_id']
    year = row['year']
    fin_material = row['produced_material']
    build_hierarchy(fin_material, fin_material, result_list, plant, year)

result_df = pd.DataFrame(result_list)
print("Total rows:", len(result_df))
print(result_df.head())
print("2025:", len(result_df[result_df['year'] == 2025]))
print("PLANT_15:", len(result_df[result_df['plant'] == 'PLANT_15']))

result_df.to_excel('bom_explosion_output.xlsx', index=False)


rows: 1332
      year  month  produced_material  produced_material_production_type  \
0     2024      1              10000                               8002   
1     2024      1              50000                               8002   
2     2024      1              50000                               8002   
3     2024      1              50000                               8002   
4     2024      1              80070                               8007   
...    ...    ...                ...                                ...   
1327  2025     12              50009                               8002   
1328  2025     12              80079                               8007   
1329  2025     12              80079                               8007   
1330  2025     12              80079                               8007   
1331  2025     12              80019                               8001   

     produced_material_release_type  produced_material_quantity  \
0                    